# Sample Solution for Lab 2

In [32]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import tensorflow as tf
from tensorflow import keras
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical

# Load the Wine dataset
from tensorflow.keras.datasets import mnist
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Split your test and train data using stratified sampling and one-hot encode them

In [33]:
encoder = OneHotEncoder(sparse=False)
y_train_encoded = to_categorical(y_train)
y_test_encoded = to_categorical(y_test)

In [34]:
# Data Preprocessing Pipeline
# I will add this to my final pipeline later
# I simply used SimpleImputer because there are no missing values in the dataset and this won't change anything, you can use better things

preprocessor = Pipeline(steps=[('imputer', SimpleImputer(strategy='mean')),('scaler', MinMaxScaler())])

# Define your keras model as a class that has fit function so that you can change the model dynamically and can also add it to the sklearn pipeline

If you check the sklearn documentation online (https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) you can see this statement: Pipeline of transforms with a final estimator. Sequentially apply a list of transforms and a final estimator. Intermediate steps of the pipeline must be ‘transforms’, that is, they must implement fit and transform methods. The final estimator only needs to implement fit. The transformers in the pipeline can be cached using memory argument.

In [35]:
class KerasClassifier(BaseEstimator, TransformerMixin):
    def __init__(self, model_func, **kwargs):
        self.model_func = model_func
        self.kwargs = kwargs
        self.model_ = None

    def fit(self, X, y):
        self.model_ = self.model_func(**self.kwargs)
        self.model_.fit(X, y, epochs=5, batch_size=32, verbose=0)
        return self

    def predict(self, X):
      # The argmax here converts the one-hot encoding to label format
        return np.argmax(self.model_.predict(X), axis=1)

# Neural Network Model with Keras
def create_model(filters, learning_rate):
    model = models.Sequential()
    model.add(layers.Conv2D(filters=filters, kernel_size=(3, 3), activation='relu',input_shape=(28, 28, 1)))
    model.add(layers.Conv2D(filters=filters, kernel_size=(3, 3), activation='relu',input_shape=(28, 28, 1)))
    model.add(layers.MaxPooling2D(pool_size=(2, 2)))
    model.add(layers.Conv2D(filters=filters, kernel_size=(3, 3), activation='relu',input_shape=(28, 28, 1)))
    model.add(layers.MaxPooling2D(pool_size=(2, 2)))
    model.add(layers.Flatten())
    model.add(layers.Dense(10, activation='softmax'))
    model.compile(optimizer='adam', loss='categorical_crossentropy',metrics=['accuracy'])
    return model

# Define your loops to do hyperparameter tunning using stratified k-fold

In [ ]:
# Hyperparameter tuning
best_f1 = 0
best_params = None
filter_grid = [16, 32]
lr_grid = [0.001, 0.01]

# Using StratifiedKFold for cross-validation
skf = StratifiedKFold(n_splits=5)

for f in filter_grid:
    for lr in lr_grid:
        f1_scores = []
        for train_idx, val_idx in skf.split(X_train, y_train):
            X_train_fold = X_train[train_idx]
            y_train_fold = y_train_encoded[train_idx]
            X_val_fold = X_train[val_idx]
            y_val_fold = y_train_encoded[val_idx]
            
            pipeline = Pipeline([
                ('classifier', KerasClassifier(create_model, filters=f, learning_rate=lr))
            ])

            # Fit and predict for current fold
            pipeline.fit(X_train_fold, y_train_fold)
            y_pred_fold = pipeline.predict(X_val_fold)
            # Convert y_val_fold from one-hot encoded to label format
            new_y_val_fold = np.argmax(y_val_fold, axis=1)

            f1 = f1_score(new_y_val_fold, y_pred_fold, average='macro')
            f1_scores.append(f1)

        # Average F1 score for the current hyperparameters
        avg_f1 = np.mean(f1_scores)
        
        if avg_f1 > best_f1:
            best_f1 = avg_f1
            best_params = {'filters': f, 'learning_rate': lr}

        print(f"Filters: {f}, Learning rate: {lr}, Avg F1 Score: {avg_f1}")

print(f"Best F1 Score: {best_f1} with parameters {best_params}")

375/375 [==============================] - 3s 7ms/step


# Train once more with the best parameters on the whole training set

In [ ]:
#Training with the best parameters
pipeline = Pipeline([
                    ('classifier', KerasClassifier(create_model, filters=32, learning_rate=0.001))
                ])

# Fit
pipeline.fit(X_train, y_train_encoded)
# Predict on the test data
y_pred = pipeline.predict(X_test)
# Convert y_test from one-hot encoded to label format
new_y_test = np.argmax(y_test_encoded, axis=1)
f1_score(new_y_test, y_pred, average='macro')